In [1]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [2]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [3]:
# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating \
this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
1.Y/N indicating whether the article is talking about a region of Boston. \n 2.The specific location within the city you got if you got Y in the first question. \
3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. PLEASE CONSIDER THE CONTEXT OF THE ARTICLE. Give your response in the following format: \
# 1. A very brief summary of what the article is talking about. \n 2.The specific location you chose based on the context of the article. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
# Headline: \n\n {headline} \n\n [/INST]""",
# )

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
# 1.Y/N indicating whether the article is talking about a region of Boston \n 2.The specific location within the city you got if you got Y in the first question. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. PLEASE KEEP YOUR ANSWER SHORT. \n\n
# Headline: \n\n {headline} \n\n Body: \n\n {body}  \n\n [/INST]""",
# )

In [4]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"

In [5]:
llm = LlamaCpp(
    model_path=llama_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u

In [6]:
chain = prompt | llm | output_parser

In [7]:
# Run LLM on a given article
def run_llm(headline, body):
    return chain.invoke({"headline": headline, "body": body})

## NER Model

In [8]:
import spacy
from span_marker import SpanMarkerModel

In [9]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [10]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [11]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [12]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [13]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
            
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [14]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

## Pipeline Entry Point

In [15]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one
# sample_data_dir = "./sample_data/se_naacp_db.articles_data.csv"

In [16]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 10 articles
raw_df = full_df.sample(10)
# raw_df = full_df
len(raw_df)


10

In [17]:
# raw_df = pd.read_csv(sample_data_path)

In [18]:
raw_df.head(10)

,_id,Type,Label,hl1,Byline,Section Navigation,Section,Title,Paths,Publish Date,Has Path?,body
5918,0000017c-568b-d44d-a57c-77cf1e440001,Article,A Boston Vet On Why Your Pet's Favorite Food M...,A Boston Vet On Why Your Pet's Favorite Food M...,Mackenzie Farkus,NaN,Lifestyle,NaN,/lifestyle/2021/10/06/one-boston-vet-on-why-yo...,Wed Oct 06 16:02:27 EDT 2021,TRUE,Pandemic-era global supply chain shortages are...
9701,00000182-373d-d99a-a5ef-f73dfb520001,Article,Happy hour drink discounts may be coming back ...,Happy hour drink discounts may be coming back ...,Fernando Cervantes Jr.,NaN,Local News,NaN,/local-news/2022/07/25/happy-hour-drink-discou...,Mon Jul 25 18:02:05 EDT 2022,TRUE,Discounted happy hour drinks have been banned ...
1349,00000176-d378-d909-af76-f77afac50001,Article,"Baker: State Has Distributed Nearly 300,000 CO...","Baker: State Has Distributed Nearly 300,000 CO...",Adam Reilly,NaN,Politics,NaN,/politics/2021/01/05/baker-state-has-distribut...,Tue Jan 05 12:33:08 EST 2021,TRUE,"At the close of the weekend, about 287,000 dos..."
8276,0000017f-e07d-d541-adff-effd48da0001,Article,"Friday, April 1","Friday, April 1",0000017f-e07d-d541-adff-effd48da0000,NaN,Digital Mural,NaN,/digital-mural/2022/04/01/friday-april-1 (Perm...,Fri Apr 01 08:43:29 EDT 2022,TRUE,Grammy-winning jazz musician and composer Tere...
10816,00000184-2f31-d335-a7ce-bf3f1fca0001,Article,It's now illegal in Mass. to throw out used je...,It's now illegal in Mass. to throw out used je...,Craig LeMoult,NaN,Local News,NaN,/local-news/2022/11/01/its-now-illegal-in-mass...,Tue Nov 01 09:58:52 EDT 2022,TRUE,It&#39;s now against the law in Massachusetts ...
11437,00000185-2b9e-d3e0-a3ef-bfbeb2040001,Article,"Boston Public Radio full show: Dec. 19, 2022","Boston Public Radio full show: Dec. 19, 2022",Brendan Deady,NaN,Local News,NaN,/local-news/2022/12/19/boston-public-radio-ful...,Mon Dec 19 15:39:18 EST 2022,TRUE,<i>We opened the show by taking our listeners’...
11146,00000184-bde5-dd99-ad94-bff53c8a0001,Article,He woke up from eye surgery with a gash on his...,He woke up from eye surgery with a gash on his...,"Fred Clasen-Kelly, Stephanie O'Neill",NaN,National News,NaN,/national-news/2022/11/28/he-woke-up-from-eye-...,Mon Nov 28 05:01:00 EST 2022,TRUE,"When Jerry Bilinski, a 67-year-old retired soc..."
12746,00000187-24bb-d281-a5f7-adbfab0a0001,Article,"Analysis: As Warren seeks a third term, her po...","Analysis: As Warren seeks a third term, her po...",Adam Reilly,NaN,Politics,NaN,/politics/2023/03/27/analysis-as-warren-seeks-...,Mon Mar 27 17:21:01 EDT 2023,TRUE,As Elizabeth Warren launches a bid for a third...
167,00000175-9d36-d944-a9fd-ddf7d40f0001,Article,CDC Report: Officials Knew Coronavirus Test Wa...,CDC Report: Officials Knew Coronavirus Test Wa...,Dina Temple-Raston,NaN,National News,NaN,/national-news/2020/11/06/cdc-report-officials...,Fri Nov 06 05:06:00 EST 2020,TRUE,"On Feb. 6, a scientist in a small infectious d..."
146,00000175-996c-d3e2-adf5-dbfc0da70001,Article,Shades of Bush v. Gore? Al Gore's Former Lawye...,Shades of Bush v. Gore? Al Gore's Former Lawye...,Meg Woolhouse,NaN,Politics,NaN,/politics/2020/11/05/shades-of-bush-v-gore-al-...,Thu Nov 05 12:51:37 EST 2020,TRUE,Maybe you hoped to never hear the words “hangi...


The ML Model honestly just needs the `id`, `header`, and `body`.

In [19]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

In [20]:
# For Testing Purposes Only
# df = df[:20]

Remove Duplicates (if any)

In [21]:
duplicates = df.duplicated(subset=['hl1'])

In [22]:
print(duplicates.value_counts())

False    10
Name: count, dtype: int64


In [23]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [24]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

100%|██████████| 10/10 [00:00<00:00, 9988.82it/s]


Clean the Body and Header with Regex

In [25]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 10/10 [00:00<00:00, 9991.20it/s]


### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary

In [26]:
known_title_locs_path = "./geodata/known_locs.json"
known_title_locs = load_cache(known_title_locs_path)

unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [27]:
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    # Look through the header for known locations
    lowercase_header = header.lower()
    for location in known_title_locs.keys():
        if (location.lower() in lowercase_header):
            if location not in unwanted_entities["FAC"]:
                return location  
    return None

## FOR TESTING: Deactivate explicit pass

In [28]:
df["Explicit_Pass"] = None

In [29]:
# df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)

In [30]:
df["Explicit_Pass"].value_counts().head(10)

Series([], Name: count, dtype: int64)

### NER Code First Pass

In [31]:
# Return the first valid facility found, or organization if none are found
def valid_facility(entities, firstPass):
    print(entities)

    if (firstPass): 
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
        else:
            return None
    
    else:
        first_org = None

        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
            
            # If it's a valid organization, save it (but don't return in case there's a facility later on)
            if (first_org == None and entity.label_ == "ORG" and entity.text not in unwanted_entities["ORG"]):
                first_org = entity.text
        else:             
            return first_org # Return regardless of whether it's None or not 


In [32]:
# Run NER on the body of the article and return first valid facility
def predict_NER_def(text, firstPass=True):
    try:
        if (text == None or text == ""):
            return None
        
        entities = nlp(text).ents
        return valid_facility(entities, firstPass)
        
    except Exception as error:
        return None

In [33]:
# Run NER on the articles that do not have an explicit location in the title
def explicit_filtering_NER(article):
    try:
        # If the article does not have an explicit location, run NER
        if (article['Explicit_Pass'] != None): 
            print(f"Has location from title: {article['hl1']}")
            return None
        else:
            return predict_NER_def(article['body'])
    except Exception as error:
        print(error)
        return None

### Attempt 1 at Chunking

Separates text into smaller chunks and runs NER into each. Then, it returns the first facility it finds.

In [34]:
# Chunk processing
chunk_size = 100
def chunk_processing(text, chunk_size=chunk_size):
    # Split text into smaller chunks
    chunks = split_text_into_chunks(text, chunk_size)

    # Process each chunk and return if a valid facilty is found
    for chunk in chunks:
        result = predict_NER_def(chunk)
        if result is not None:
             return result
    return None

# Split article text into chunks of specified size
def split_text_into_chunks(text, chunk_size=chunk_size):
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

In [ ]:
def handle_chunk_processing(article):
    # Filter out articles that have an explicit location in the title
    if (article['Explicit_Pass'] != None): 
        print(f"Has location from title: {article['hl1']}")
        return None

    # Filter out articles that have no text
    text = article['body']
    if (text is None or text == ""):
            return None
    try:
        return chunk_processing(text)
    except Exception as error:
        print(error)

In [35]:
import time
# Run NER in batches
start_time = time.time()
# df["NER_Pass_Chunk"] = df.progress_apply(handle_chunk_processing, axis=1)
total_time = time.time() - start_time
print(f"Total Time: {total_time}")


Total Time: 0.0


In [36]:
df

,_id,hl1,body,Explicit_Pass
5918,0000017c-568b-d44d-a57c-77cf1e440001,Boston Vet On Why Your Pet Favorite Food Might...,Pandemic era global supply chain shortages are...,None
9701,00000182-373d-d99a-a5ef-f73dfb520001,Happy hour drink discounts may be coming back ...,Discounted happy hour drinks have been banned ...,None
1349,00000176-d378-d909-af76-f77afac50001,Baker State Has Distributed Nearly 300 000 COV...,At the close of the weekend about 287 000 dose...,None
8276,0000017f-e07d-d541-adff-effd48da0001,Friday April,Grammy winning jazz musician and composer Tere...,None
10816,00000184-2f31-d335-a7ce-bf3f1fca0001,It now illegal in Mass. to throw out used jean...,It now against the law in Massachusetts to thr...,None
11437,00000185-2b9e-d3e0-a3ef-bfbeb2040001,Boston Public Radio full show Dec. 19 2022,We opened the show by taking our listeners cal...,None
11146,00000184-bde5-dd99-ad94-bff53c8a0001,He woke up from eye surgery with gash on his f...,When Jerry Bilinski 67 year old retired social...,None
12746,00000187-24bb-d281-a5f7-adbfab0a0001,Analysis As Warren seeks third term her positi...,As Elizabeth Warren launches bid for third ter...,None
167,00000175-9d36-d944-a9fd-ddf7d40f0001,CDC Report Officials Knew Coronavirus Test Was...,On Feb. scientist in small infectious disease ...,None
146,00000175-996c-d3e2-adf5-dbfc0da70001,Shades of Bush v. Gore Al Gore Former Lawyer A...,Maybe you hoped to never hear the words hangin...,None


## MultiProcessing Attempt

In [39]:
# Chunk processing
chunk_size = 100
def chunk_processing(text, chunk_size=chunk_size):
    if (text is None or text == ""):
            return None

    # Split text into smaller chunks
    chunks = split_text_into_chunks(text, chunk_size)
    
    # Process each chunk and return if a valid facilty is found
    for chunk in chunks:
        result = predict_NER_def(chunk)
        if result is not None:
             return result
    return None


In [ ]:
def valid_facility2(entities):
    for entity in entities:
        # If it's a valid facility, return it
        if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
            print("valid facility", entity.text)
            return entity.text
    else:
        return None


# Run NER on the body of the article and return first valid facility
def predict_NER_def2(text):
    try:
        if (text == None or text == ""):
            return None
        
        entities = nlp(text).ents
        return valid_facility2(entities)
        
    except Exception as error:
        return error

In [ ]:
import ipyparallel as ipp
import multiprocessing as mp

# Start and connect to an IPyParallel cluster
rc = ipp.Cluster(n=mp.cpu_count() - 2).start_and_connect_sync()
dview = rc[:]

# Process the articles parallelly using ipyparallel
def process_articles(data):
    text_list = data['body'].tolist()
    
    print("Collecting results from ipyparallel map...")
    results = dview.map_sync(predict_NER_def2, text_list)
    return results

# Process the articles using ipyparallel
results = process_articles(df)
print(results)

Starting 10 engines with <class 'ipyparallel.cluster.launcher.LocalEngineSetLauncher'>


  0%|          | 0/10 [00:00<?, ?engine/s]

[None, None, None, None, None, None, None, None, None, None]


Failed to remove C:\Users\axel0\.ipython\profile_default\log\ipengine-1720474076-pjfw-1720474077-7.log: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\axel0\\.ipython\\profile_default\\log\\ipengine-1720474076-pjfw-1720474077-7.log'
Output for 7:
2024-07-08 15:27:59.669 [IPEngine] Loading connection info from $IPP_CONNECTION_INFO
2024-07-08 15:27:59.669 [IPEngine] WARNING | Not using CurveZMQ security
2024-07-08 15:27:59.675 [IPEngine] Registering with controller at tcp://127.0.0.1:64011
2024-07-08 15:27:59.693 [IPEngine] Shell_addrs: ['tcp://127.0.0.1:64013', 'tcp://127.0.0.1:64014', 'tcp://127.0.0.1:64019']
2024-07-08 15:27:59.695 [IPEngine] Connecting shell to tcp://127.0.0.1:64013
2024-07-08 15:27:59.695 [IPEngine] Connecting shell to tcp://127.0.0.1:64014
2024-07-08 15:27:59.695 [IPEngine] Connecting shell to tcp://127.0.0.1:64019
2024-07-08 15:27:59.696 [IPEngine] Starting nanny
2024-07-08 15:28:00.906 [KernelNanny.7] Sta

In [ ]:
# import multiprocessing as mp

# def process_articles(data):    
#     with mp.Pool(mp.cpu_count() - 2) as pool:
#         print("Collecting results from tqdm iterator...")
#         results = pool.map(test_function, ["hi", "hello", "bye"])
#     return results

In [ ]:
#  results = process_articles(df)

In [ ]:
# import multiprocessing as mp

# def process_articles(data):
#     # Filter out articles that have an explicit location in the title
    
#     with mp.Pool(mp.cpu_count() - 2) as pool:
#         # Step 4: Create an iterator for the pool.imap function
#         # pool_imap_iterator = pool.imap(chunk_processing, data['body'])

#         # Step 5: Wrap the iterator with tqdm to display a progress bar
#         # tqdm_iterator = tqdm(pool_imap_iterator, total=len(data), desc="Processing Articles")

#         # Step 6: Convert the tqdm iterator to a list to collect results
#         print("Collecting results from tqdm iterator...")
#         # results = pool.map(chunk_processing, data)
#         results = pool.map(test_function, data['body'])
#         # results = []
#         # for result in tqdm_iterator:
#         #     results.append(result)
#         #     print(f"Processed article, current results length: {len(results)}")
#         # results = list(tqdm(pool.imap(partial(chunk_processing, chunk_size=chunk_size), data), total=len(data)))
#     return results

In [ ]:
# results = process_articles(df)

In [ ]:
import multiprocessing as mp

print("Number of processors: ", mp.cpu_count())

def parallel_ner(articles_df, num_processes=(2)):
    articles = articles_df.to_dict('records')

    # Create a multiprocessing pool with the specified number of processes
    with mp.Pool(processes=num_processes) as pool:
        # Initialize the tqdm progress bar
        progress_bar = tqdm(total=len(articles))

        # Define a generator that updates the progress bar
        def imap_generator():
            for result in pool.imap(explicit_filtering_NER, articles):
                yield result
                progress_bar.update(1)

        # Collect all results using the generator
        results = list(imap_generator())

        progress_bar.close()
    
    return results

# Run NER in parallel
# df["NER_Pass_"] = parallel_ner(df)

In [ ]:
start_time = time.time()
df['NER_Pass'] = df.progress_apply(explicit_filtering_NER, axis=1)
total_time = time.time() - start_time
print(f"Total Time: {total_time}")


In [ ]:
df.head(10)

### Llama Prediction

In [ ]:
#TODO: Comply with token limit of 2048 for Llama
# Run the LLM model on the articles that haven't been tagged with a location yet. Then run NER on the LLM prediction
def predict_llama(article):
    try:
        # If the article does not have an explicit location or NER location, run LLM
        if (article['Explicit_Pass'] != None or article['NER_Pass'] != None):
            print(f"Has location from title or NER: {article['hl1']}")
            return None
        else:
            llama_prediction = run_llm(article['hl1'], article['body'])
            print(llama_prediction)
            return predict_NER_def(llama_prediction, True)
    except Exception as error:
        print(error)
        return None

In [ ]:
df['NER_Prediction'] = df.progress_apply(predict_llama, axis=1)

In [ ]:
df.head(10)

In [ ]:
## TODO: DELETE AFTER POPULATING THE UNWANTED ENTITIES CACHE
# unwanted_entities = {
#     'FAC': ['Boston'],
#     'ORG': ['New York Times'],
#     'LOC': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
#     'GPE': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
# }

# save_cache_to_file(unwanted_entities, unwanted_entities_path)

Extract locations from the most specific pass

In [ ]:
# Get the locations from the most specific pass for a given article
def extractLocations(article):
    for key in ['Explicit_Pass', 'NER_Pass', 'NER_Prediction']:
        location = article.get(key)
        if location is not None:
            return location
    return None

In [ ]:
df['Locations'] = df.progress_apply(extractLocations, axis=1)

In [ ]:
df.head(10)

In [ ]:
df

## Get the coordinates

In [ ]:
known_locations_path = "./geodata/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [ ]:
# Get the coordinates of the location
def getCoordinates(location): # Valid labels are FAC for NER_Pass; FAC and ORG for NER_Prediction
    if (location == None or len(location) == 0): return None  
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [ ]:
df['Coordinates'] = df['Locations'].progress_apply(getCoordinates)

In [ ]:
df.head(10)

## Geocode locations

In [ ]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + location)
        except KeyError:
            print("Location is outside of the United States: " + location)
        except Exception as error:
            print(error)

    return None, None  # Return this if API call failed or no tracts found

In [ ]:
# Get the census tract and county of the location
def geocode(location):
    if (location is None or len(location) == 0): return None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"]
    County = known_locations[location]["county"]
    if (Tract is None or County is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County
    

In [ ]:
df[['Tracts', 'County']] = df.progress_apply(lambda row: pd.Series(geocode(row['Locations'])), axis=1)
df

In [ ]:
print(df['Explicit_Pass'].value_counts().sum())
df['Explicit_Pass'].value_counts()

In [ ]:
print(df['NER_Pass'].value_counts().sum())
df['NER_Pass'].value_counts()

In [ ]:
print(df['NER_Prediction'].value_counts().sum())
df['NER_Prediction'].value_counts()

In [ ]:
df

In [ ]:
df.head(10)

In [ ]:
len(df)

In [ ]:
df = df.dropna(subset=["Tracts", "County"]) # Clean those that don't have a Tract or a County

In [ ]:
print(len(df))
df.head(10)

## Topic Modeling

In [ ]:
import os
import tiktoken
import numpy as np
from transformers import pipeline
from sklearn.metrics import adjusted_rand_score
from openai import OpenAI, AsyncOpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

## OpenAI Client

In [ ]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

In [ ]:
client = OpenAI(
    api_key='YOUR_KEY_HERE',
)

## Taxonomy Lists

Content Taxanomy

In [ ]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./taxonomy_list/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

Selected Taxonomy List

In [ ]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./topics/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [ ]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./topics/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

## Obtaining Ada Embedding

In [ ]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

In [ ]:
df['topic_model_body'] = df['body'].apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
df['tokens'] = df['topic_model_body'].apply(lambda x: x.split())
df['tokens'] = df['tokens'].apply(truncate)

In [ ]:
df['ada_embedding'] = df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

## Similarity Matching After Ada Embedding

In [ ]:
# Find most similar taxonomy (out of all toipcs) to news body
closest_topic_list_all = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = all_topics_list[closest_topic_index]
    closest_topic_list_all.append(closest_topic)

df['closest_topic_all'] = closest_topic_list_all

In [ ]:
# Find most similar taxonomy (out of 230 selected topics) to news body
closest_topic_list_selected = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = selected_topics_list[closest_topic_index]
    closest_topic_list_selected.append(closest_topic)

df['closest_topic_selected'] = closest_topic_list_selected

In [ ]:
client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
client_topic_list = client_taxonomy_df['label'].to_list()
similarity_arr = []

closest_topic_list_client = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
    
    if max(similarities) > 0.25:    
        closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
        closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
        closest_topic_list_client.append(closest_topic)
    else:
        closest_topic_list_client.append('Other')
    similarity_arr.append(max(similarities))
    
df['closest_topic_client'] = closest_topic_list_client

In [ ]:
df

In [ ]:
df.to_csv("./outputs/gbh_output.csv")

In [ ]:
raw_df

In [ ]:
df

In [ ]:
merged_df = pd.merge(raw_df, df, on='_id', how='inner')

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("./outputs/gbh_output_all_fields.csv")

In [ ]:
merged_df.columns